In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


DATA_PATH = Path("used_cars_cleaned_08.csv")
SEED = 42

df = pd.read_csv(DATA_PATH)

# Match the selected Ridge configuration, excluding emission_class.
numeric_cols = [
    "mileage_miles", "reg_year", "power_hp", "previous_owners",
    "annual_tax_gbp", "co2_gmile",
]
count_cols = ["doors", "seats"]
categorical_cols = [
    "make", "model", "version", "fuel", "transmission", "body_type",
    "body_color", "drivetrain", "seller_type", "seller_city", "seller_county",
]

X = df[numeric_cols + count_cols + categorical_cols].copy()
y = df["price_gbp"]

# Keep rows with identical model inputs in the same split.
groups = pd.util.hash_pandas_object(X, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(splitter.split(X, y, groups=groups))


def add_features(frame):
    return frame.assign(
        log_mileage=np.log1p(frame["mileage_miles"]),
        make_model=frame["make"] + " | " + frame["model"],
    )


X_train = add_features(X.iloc[train_idx])
X_val = add_features(X.iloc[val_idx])
y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

print(f"Training listings: {len(X_train):,}")
print(f"Validation listings: {len(X_val):,}")

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols + ["log_mileage"]),
    ("counts", SimpleImputer(strategy="most_frequent"), count_cols),
    (
        "categories",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_cols + ["make_model"],
    ),
])

ridge = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("scale", StandardScaler(with_mean=False)),
        ("model", Ridge(alpha=0.1, solver="lsqr")),
    ]),
    func=np.log1p,
    inverse_func=np.expm1,
)

forest = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", RandomForestRegressor(
            n_estimators=40,
            max_depth=24,
            min_samples_leaf=3,
            max_features=0.7,
            max_samples=0.6,
            random_state=SEED,
            n_jobs=2,
            verbose=1,
        )),
    ]),
    func=np.log1p,
    inverse_func=np.expm1,
)

actual = y_val.to_numpy()
price_cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, price_cutoffs)
results = []

for name, model in [
    ("Refit log-price Ridge", ridge),
    ("Log-price Random Forest", forest),
]:
    print(f"Training {name}...", flush=True)
    model.fit(X_train, y_train)

    predicted = model.predict(X_val)
    absolute_error = np.abs(actual - predicted)
    predicted_bands = np.digitize(predicted, price_cutoffs)

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(actual_bands == predicted_bands),
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(3))

Training listings: 289,272
Validation listings: 72,318
Training Refit log-price Ridge...
Training Log-price Random Forest...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed: 21.6min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.8s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Refit log-price Ridge,2239.498,72044.113,-3.122,71.869,93.999
Log-price Random Forest,1850.264,24304.879,0.531,68.072,93.451


In [2]:
from time import perf_counter

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

# Create a tuning split within X_train, keeping identical inputs together.
groups = pd.util.hash_pandas_object(X_train, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
fit_idx, tune_idx = next(splitter.split(X_train, y_train, groups=groups))

# Use up to 60,000 fitting rows for each trial to limit runtime.
rng = np.random.default_rng(42)
if len(fit_idx) > 60_000:
    fit_idx = np.sort(rng.choice(fit_idx, size=60_000, replace=False))

X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
X_tune, y_tune = X_train.iloc[tune_idx], y_train.iloc[tune_idx]

trials = [
    {"min_samples_leaf": 3, "max_features": 0.7},  # Current settings
    {"min_samples_leaf": 2, "max_features": 0.7},  # Smaller leaves
    {"min_samples_leaf": 3, "max_features": 0.4},  # Fewer features per split
]

trial_results = []
best_mae = np.inf
best_settings = None

for settings in trials:
    model_params = {
        f"regressor__model__{name}": value
        for name, value in settings.items()
    }
    candidate = clone(forest).set_params(
        regressor__model__n_estimators=15,
        regressor__model__verbose=0,
        **model_params,
    )

    start = perf_counter()
    candidate.fit(X_fit, y_fit)
    mae = mean_absolute_error(y_tune, candidate.predict(X_tune))
    elapsed = perf_counter() - start

    trial_results.append({
        **settings,
        "Tuning MAE (£)": mae,
        "Seconds": elapsed,
    })
    print(f"{settings} | MAE £{mae:,.2f} | {elapsed:.0f} sec", flush=True)

    if mae < best_mae:
        best_mae = mae
        best_settings = model_params

display(pd.DataFrame(trial_results).sort_values("Tuning MAE (£)").round(2))

# Reuse the fitted forest if its settings win.
if best_settings == {
    "regressor__model__min_samples_leaf": 3,
    "regressor__model__max_features": 0.7,
}:
    tuned_forest = forest
    print("The current settings won. Reusing the fitted forest.")
else:
    print(f"Refitting 40 trees with {best_settings}", flush=True)
    tuned_forest = clone(forest).set_params(**best_settings)
    tuned_forest.fit(X_train, y_train)

actual = y_val.to_numpy()
actual_bands = np.digitize(actual, [10_000, 25_000])
results = []

for name, model in [("Current forest", forest), ("Selected forest", tuned_forest)]:
    predicted = model.predict(X_val)
    error = np.abs(actual - predicted)

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(
            np.digitize(predicted, [10_000, 25_000]) == actual_bands
        ),
    })

display(pd.DataFrame(results).set_index("Model").round(3))

{'min_samples_leaf': 3, 'max_features': 0.7} | MAE £2,383.31 | 45 sec
{'min_samples_leaf': 2, 'max_features': 0.7} | MAE £2,325.89 | 53 sec
{'min_samples_leaf': 3, 'max_features': 0.4} | MAE £2,451.97 | 28 sec


,min_samples_leaf,max_features,Tuning MAE (£),Seconds
1,2,0.7,2325.89,52.69
0,3,0.7,2383.31,45.41
2,3,0.4,2451.97,27.84


Refitting 40 trees with {'regressor__model__min_samples_leaf': 2, 'regressor__model__max_features': 0.7}


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed: 23.2min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.6s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.6s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Current forest,1850.264,24304.879,0.531,68.072,93.451
Selected forest,1802.159,23166.157,0.574,68.589,93.519


In [3]:
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

rows_per_tree = min(15_000, len(X_train))
actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)

models = {}
results = []

for name, bootstrap in [("Bagging", True), ("Pasting", False)]:
    ensemble = BaggingRegressor(
        estimator=DecisionTreeRegressor(
            max_depth=18,
            min_samples_leaf=2,
        ),
        n_estimators=10,
        max_samples=rows_per_tree,
        bootstrap=bootstrap,
        random_state=42,
        n_jobs=2,
        verbose=1,
    )

    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", ensemble),
        ]),
        func=np.log1p,
        inverse_func=np.expm1,
    )

    print(f"Training {name} with {rows_per_tree:,} rows drawn per tree...", flush=True)
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - start

    predicted = model.predict(X_val)
    absolute_error = np.abs(actual - predicted)
    predicted_bands = np.digitize(predicted, cutoffs)

    models[name] = model
    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(actual_bands == predicted_bands),
        "Fit time (sec)": fit_seconds,
    })

comparison = pd.DataFrame(results).set_index("Model").sort_values("MAE (£)")
display(comparison.round(3))

best_ensemble = models[comparison.index[0]]

Training Bagging with 15,000 rows drawn per tree...


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:  1.0min finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


Training Pasting with 15,000 rows drawn per tree...


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.2s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:   59.7s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.2s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%),Fit time (sec)
Model,,,,,,
Pasting,2637.344,26802.553,0.430,55.19,91.037,61.304
Bagging,2652.760,26743.986,0.432,55.01,90.915,64.668


In [4]:
from itertools import product
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor


SEED = 42
MAX_TUNING_FIT_ROWS = 60_000
MAX_TUNING_SCORE_ROWS = 20_000
TRIAL_TREES = 8
TRIAL_ROWS_PER_TREE = 12_000
FINAL_TREES = 20
FINAL_ROWS_PER_TREE = 20_000
FOREST_MAE = 1816.211

# Split within X_train so X_val stays separate from tuning.
groups = pd.util.hash_pandas_object(X_train, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
fit_idx, tune_idx = next(splitter.split(X_train, y_train, groups=groups))

rng = np.random.default_rng(SEED)

if len(fit_idx) > MAX_TUNING_FIT_ROWS:
    fit_idx = np.sort(rng.choice(fit_idx, MAX_TUNING_FIT_ROWS, replace=False))

if len(tune_idx) > MAX_TUNING_SCORE_ROWS:
    tune_idx = np.sort(rng.choice(tune_idx, MAX_TUNING_SCORE_ROWS, replace=False))

X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
X_tune, y_tune = X_train.iloc[tune_idx], y_train.iloc[tune_idx]


def build_ensemble(bootstrap, depth, leaf_size, n_trees, rows_per_tree, verbose=0):
    ensemble = BaggingRegressor(
        estimator=DecisionTreeRegressor(
            max_depth=depth,
            min_samples_leaf=leaf_size,
        ),
        n_estimators=n_trees,
        max_samples=rows_per_tree,
        bootstrap=bootstrap,
        random_state=SEED,
        n_jobs=2,
        verbose=verbose,
    )

    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", ensemble),
        ]),
        func=np.log1p,
        inverse_func=np.expm1,
    )


methods = {"Bagging": True, "Pasting": False}
settings = list(product([18, 24], [2, 5]))
trial_results = []

for name, bootstrap in methods.items():
    for depth, leaf_size in settings:
        candidate = build_ensemble(
            bootstrap=bootstrap,
            depth=depth,
            leaf_size=leaf_size,
            n_trees=TRIAL_TREES,
            rows_per_tree=min(TRIAL_ROWS_PER_TREE, len(X_fit)),
        )

        start = perf_counter()
        candidate.fit(X_fit, y_fit)
        mae = mean_absolute_error(y_tune, candidate.predict(X_tune))
        elapsed = perf_counter() - start

        trial_results.append({
            "Method": name,
            "max_depth": depth,
            "min_samples_leaf": leaf_size,
            "Tuning MAE (£)": mae,
            "Seconds": elapsed,
        })

        print(
            f"{name} | depth={depth}, leaf={leaf_size} | "
            f"MAE £{mae:,.2f} | {elapsed:.1f}s",
            flush=True,
        )

tuning_results = pd.DataFrame(trial_results).sort_values(
    ["Method", "Tuning MAE (£)"]
)
display(tuning_results.round(2))

# Refit the best configuration for each method on the training split.
winners = tuning_results.drop_duplicates("Method", keep="first")
actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)

tuned_ensembles = {}
validation_results = []

for _, winner in winners.iterrows():
    name = winner["Method"]
    depth = int(winner["max_depth"])
    leaf_size = int(winner["min_samples_leaf"])

    model = build_ensemble(
        bootstrap=methods[name],
        depth=depth,
        leaf_size=leaf_size,
        n_trees=FINAL_TREES,
        rows_per_tree=min(FINAL_ROWS_PER_TREE, len(X_train)),
        verbose=1,
    )

    print(f"Refitting {name} | depth={depth}, leaf={leaf_size}", flush=True)
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - start

    predicted = model.predict(X_val)
    mae = mean_absolute_error(actual, predicted)
    tuned_ensembles[name] = model

    validation_results.append({
        "Method": name,
        "max_depth": depth,
        "min_samples_leaf": leaf_size,
        "MAE (£)": mae,
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(
            np.abs(actual - predicted) <= 0.10 * actual
        ),
        "Band accuracy (%)": 100 * np.mean(
            np.digitize(predicted, cutoffs) == actual_bands
        ),
        "MAE gain vs forest (£)": FOREST_MAE - mae,
        "Fit time (sec)": fit_seconds,
    })

validation_comparison = (
    pd.DataFrame(validation_results)
    .set_index("Method")
    .sort_values("MAE (£)")
)
display(validation_comparison.round(3))

best_ensemble = tuned_ensembles[validation_comparison.index[0]]

Bagging | depth=18, leaf=2 | MAE £2,571.03 | 14.5s
Bagging | depth=18, leaf=5 | MAE £2,784.54 | 8.7s
Bagging | depth=24, leaf=2 | MAE £2,553.96 | 15.1s
Bagging | depth=24, leaf=5 | MAE £2,780.64 | 8.4s
Pasting | depth=18, leaf=2 | MAE £2,552.46 | 14.5s
Pasting | depth=18, leaf=5 | MAE £2,768.52 | 9.6s
Pasting | depth=24, leaf=2 | MAE £2,519.44 | 16.9s
Pasting | depth=24, leaf=5 | MAE £2,765.25 | 9.9s


,Method,max_depth,min_samples_leaf,Tuning MAE (£),Seconds
2,Bagging,24,2,2553.96,15.12
0,Bagging,18,2,2571.03,14.53
3,Bagging,24,5,2780.64,8.42
1,Bagging,18,5,2784.54,8.74
6,Pasting,24,2,2519.44,16.90
4,Pasting,18,2,2552.46,14.48
7,Pasting,24,5,2765.25,9.89
5,Pasting,18,5,2768.52,9.56


Refitting Bagging | depth=24, leaf=2


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:  3.0min finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


Refitting Pasting | depth=24, leaf=2


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.5s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:  2.9min finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.6s finished


,max_depth,min_samples_leaf,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%),MAE gain vs forest (£),Fit time (sec)
Method,,,,,,,,,
Pasting,24,2,2406.249,26614.072,0.438,59.350,91.840,-590.038,173.819
Bagging,24,2,2433.219,26515.785,0.442,59.284,91.797,-617.008,180.545


In [5]:
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

extra_trees = clone(forest).set_params(
    regressor__model=ExtraTreesRegressor(
        n_estimators=20,
        max_depth=24,
        min_samples_leaf=2,
        max_features=0.7,
        bootstrap=True,
        max_samples=0.3,
        random_state=42,
        n_jobs=2,
        verbose=2,
    )
)

print("Training Extra Trees...", flush=True)
start = perf_counter()
extra_trees.fit(X_train, y_train)
fit_seconds = perf_counter() - start

minutes, seconds = divmod(round(fit_seconds), 60)
print(f"Extra Trees fit time: {minutes} min {seconds:02d} sec")

actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)
results = []

for name, model in [
    ("Random Forest", forest),
    ("Extra Trees", extra_trees),
]:
    predicted = model.predict(X_val)
    absolute_error = np.abs(actual - predicted)

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(
            np.digitize(predicted, cutoffs) == actual_bands
        ),
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(3))

mae_gain = (
    comparison.loc["Random Forest", "MAE (£)"]
    - comparison.loc["Extra Trees", "MAE (£)"]
)
print(f"Extra Trees MAE improvement: £{mae_gain:+,.2f}")

Training Extra Trees...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.


building tree 1 of 20
building tree 2 of 20
building tree 3 of 20
building tree 4 of 20
building tree 5 of 20
building tree 6 of 20
building tree 7 of 20
building tree 8 of 20
building tree 9 of 20
building tree 10 of 20
building tree 11 of 20
building tree 12 of 20
building tree 13 of 20
building tree 14 of 20
building tree 15 of 20
building tree 16 of 20
building tree 17 of 20
building tree 18 of 20
building tree 19 of 20
building tree 20 of 20


[Parallel(n_jobs=2)]: Done  20 out of  20 | elapsed:  5.9min finished


Extra Trees fit time: 5 min 55 sec


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.5s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  20 out of  20 | elapsed:    0.1s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Random Forest,1850.264,24304.879,0.531,68.072,93.451
Extra Trees,2001.191,24934.983,0.506,65.123,92.767


Extra Trees MAE improvement: £-150.93


In [6]:
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

# Tune within X_train. Keep identical inputs together.
groups = pd.util.hash_pandas_object(X_train, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
fit_idx, tune_idx = next(splitter.split(X_train, y_train, groups=groups))

rng = np.random.default_rng(SEED)
fit_idx = np.sort(rng.choice(fit_idx, min(60_000, len(fit_idx)), replace=False))
tune_idx = np.sort(rng.choice(tune_idx, min(20_000, len(tune_idx)), replace=False))

X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
X_tune, y_tune = X_train.iloc[tune_idx], y_train.iloc[tune_idx]

trials = [
    {"max_depth": 24, "min_samples_leaf": 2, "max_features": 0.7},  # Reference
    {"max_depth": 30, "min_samples_leaf": 2, "max_features": 0.7},
    {"max_depth": 24, "min_samples_leaf": 1, "max_features": 0.7},
    {"max_depth": 24, "min_samples_leaf": 2, "max_features": 0.4},
]

def build_extra_trees(settings, n_trees, rows_per_tree, verbose=0):
    params = {f"regressor__model__{key}": value for key, value in settings.items()}
    return clone(extra_trees).set_params(
        regressor__model__n_estimators=n_trees,
        regressor__model__max_samples=rows_per_tree,
        regressor__model__verbose=verbose,
        **params,
    )

trial_results = []

for trial_number, settings in enumerate(trials, start=1):
    candidate = build_extra_trees(settings, 8, min(15_000, len(X_fit)))

    start = perf_counter()
    candidate.fit(X_fit, y_fit)
    mae = mean_absolute_error(y_tune, candidate.predict(X_tune))
    elapsed = perf_counter() - start

    trial_results.append({
        "Trial": trial_number,
        **settings,
        "Tuning MAE (£)": mae,
        "Seconds": elapsed,
    })
    print(f"Trial {trial_number}/{len(trials)} | MAE £{mae:,.2f} | {elapsed:.1f}s",
          flush=True)

tuning_results = pd.DataFrame(trial_results).sort_values("Tuning MAE (£)")
display(tuning_results.round(3))

best_trial = int(tuning_results.iloc[0]["Trial"])
best_settings = trials[best_trial - 1]
final_sample_size = min(40_000, len(X_train))

print(f"Refitting 20 trees with {best_settings}", flush=True)
tuned_extra_trees = build_extra_trees(best_settings, 20, final_sample_size, verbose=2)

start = perf_counter()
tuned_extra_trees.fit(X_train, y_train)
minutes, seconds = divmod(round(perf_counter() - start), 60)
print(f"Refit time: {minutes} min {seconds:02d} sec")

actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)
results = []

for name, model in [
    ("Random Forest", forest),
    ("Original Extra Trees", extra_trees),
    ("Tuned Extra Trees", tuned_extra_trees),
]:
    predicted = model.predict(X_val)
    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(
            np.abs(actual - predicted) <= 0.10 * actual
        ),
        "Band accuracy (%)": 100 * np.mean(
            np.digitize(predicted, cutoffs) == actual_bands
        ),
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.sort_values("MAE (£)").round(3))

Trial 1/4 | MAE £2,544.37 | 12.6s
Trial 2/4 | MAE £2,594.52 | 12.3s
Trial 3/4 | MAE £2,550.76 | 18.0s
Trial 4/4 | MAE £2,754.41 | 7.5s


,Trial,max_depth,min_samples_leaf,max_features,Tuning MAE (£),Seconds
0,1,24,2,0.7,2544.367,12.621
2,3,24,1,0.7,2550.762,17.989
1,2,30,2,0.7,2594.520,12.336
3,4,24,2,0.4,2754.413,7.453


Refitting 20 trees with {'max_depth': 24, 'min_samples_leaf': 2, 'max_features': 0.7}


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.


building tree 1 of 20building tree 2 of 20

building tree 3 of 20
building tree 4 of 20
building tree 5 of 20
building tree 6 of 20
building tree 7 of 20
building tree 8 of 20
building tree 9 of 20
building tree 10 of 20
building tree 11 of 20
building tree 12 of 20
building tree 13 of 20
building tree 14 of 20
building tree 15 of 20
building tree 16 of 20
building tree 17 of 20
building tree 18 of 20
building tree 19 of 20
building tree 20 of 20


[Parallel(n_jobs=2)]: Done  20 out of  20 | elapsed:  3.3min finished


Refit time: 3 min 19 sec


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.4s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  20 out of  20 | elapsed:    0.1s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  20 out of  20 | elapsed:    0.1s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Random Forest,1850.264,24304.879,0.531,68.072,93.451
Original Extra Trees,2001.191,24934.983,0.506,65.123,92.767
Tuned Extra Trees,2232.131,26227.384,0.454,61.496,92.150


In [7]:
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

SEED = 42
MAX_TRAIN_ROWS = 60_000

# Sample training rows only. Keep the full validation split for scoring.
rng = np.random.default_rng(SEED)
fit_idx = np.sort(
    rng.choice(len(X_train), size=min(MAX_TRAIN_ROWS, len(X_train)), replace=False)
)
X_ada_train = X_train.iloc[fit_idx]
y_ada_train = y_train.iloc[fit_idx]

adaboost = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(
                max_depth=6,
                min_samples_leaf=10,
                random_state=SEED,
            ),
            n_estimators=30,
            learning_rate=0.1,
            loss="linear",
            random_state=SEED,
        )),
    ]),
    func=np.log1p,
    inverse_func=np.expm1,
)

print(f"Training AdaBoost on {len(X_ada_train):,} listings...", flush=True)
start = perf_counter()
adaboost.fit(X_ada_train, y_ada_train)
minutes, seconds = divmod(round(perf_counter() - start), 60)
print(f"AdaBoost fit time: {minutes} min {seconds:02d} sec")

actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)
results = []

for name, model in [
    ("Random Forest", forest),
    ("AdaBoost", adaboost),
]:
    predicted = model.predict(X_val)
    absolute_error = np.abs(actual - predicted)

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(
            np.digitize(predicted, cutoffs) == actual_bands
        ),
    })

display(pd.DataFrame(results).set_index("Model").round(3))

Training AdaBoost on 60,000 listings...
AdaBoost fit time: 0 min 32 sec


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.4s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Random Forest,1850.264,24304.879,0.531,68.072,93.451
AdaBoost,4242.299,21524.890,0.632,32.372,84.227


In [8]:
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

# Keep identical predictor rows together during tuning.
groups = pd.util.hash_pandas_object(X_ada_train, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
fit_idx, tune_idx = next(
    splitter.split(X_ada_train, y_ada_train, groups=groups)
)

X_fit, y_fit = X_ada_train.iloc[fit_idx], y_ada_train.iloc[fit_idx]
X_tune, y_tune = X_ada_train.iloc[tune_idx], y_ada_train.iloc[tune_idx]

trials = {
    "Original settings": {},
    "Depth 10": {"estimator__max_depth": 10},
    "Depth 14": {"estimator__max_depth": 14},
    "Depth 10, faster learning": {
        "estimator__max_depth": 10,
        "learning_rate": 0.3,
    },
    "Depth 10, more trees": {
        "estimator__max_depth": 10,
        "n_estimators": 50,
    },
}

best_name = None
best_mae = np.inf
tuning_rows = []

for name, settings in trials.items():
    params = {
        f"regressor__model__{key}": value
        for key, value in settings.items()
    }
    candidate = clone(adaboost).set_params(**params)

    start = perf_counter()
    candidate.fit(X_fit, y_fit)
    mae = mean_absolute_error(y_tune, candidate.predict(X_tune))
    elapsed = perf_counter() - start

    tuning_rows.append({
        "Trial": name,
        "Tuning MAE (£)": mae,
        "Fit + score (sec)": elapsed,
    })
    print(f"{name}: £{mae:,.2f} in {elapsed:.1f}s", flush=True)

    if mae < best_mae:
        best_name = name
        best_mae = mae
        best_params = params

display(pd.DataFrame(tuning_rows).sort_values("Tuning MAE (£)").round(2))
print(f"Selected trial: {best_name}")

# Reuse the original fit if its settings win.
tuned_adaboost = adaboost
if best_params:
    print("Refitting selected AdaBoost on the original training sample...", flush=True)
    tuned_adaboost = clone(adaboost).set_params(**best_params)
    tuned_adaboost.fit(X_ada_train, y_ada_train)

actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)
validation_rows = []

for name, model in [
    ("Random Forest", forest),
    ("Original AdaBoost", adaboost),
    ("Selected AdaBoost", tuned_adaboost),
]:
    predicted = model.predict(X_val)
    error = np.abs(actual - predicted)

    validation_rows.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(
            np.digitize(predicted, cutoffs) == actual_bands
        ),
    })

display(
    pd.DataFrame(validation_rows)
    .set_index("Model")
    .sort_values("MAE (£)")
    .round(3)
)

Original settings: £4,220.42 in 34.1s
Depth 10: £3,098.94 in 78.2s
Depth 14: £2,600.39 in 138.2s
Depth 10, faster learning: £3,035.40 in 69.4s
Depth 10, more trees: £3,052.26 in 124.1s


,Trial,Tuning MAE (£),Fit + score (sec)
2,Depth 14,2600.39,138.17
3,"Depth 10, faster learning",3035.40,69.44
4,"Depth 10, more trees",3052.26,124.13
1,Depth 10,3098.94,78.22
0,Original settings,4220.42,34.13


Selected trial: Depth 14
Refitting selected AdaBoost on the original training sample...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    1.1s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Random Forest,1850.264,24304.879,0.531,68.072,93.451
Selected AdaBoost,2630.020,19413.520,0.701,52.062,90.242
Original AdaBoost,4242.299,21524.890,0.632,32.372,84.227


In [9]:
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reuse AdaBoost's preprocessing and log-price target transformation.
gradient_boosting = clone(adaboost).set_params(
    regressor__model=GradientBoostingRegressor(
        n_estimators=60,
        learning_rate=0.1,
        max_depth=5,
        min_samples_leaf=10,
        random_state=42,
        verbose=1,
    )
)

print(f"Training Gradient Boosting on {len(X_ada_train):,} listings...", flush=True)
start = perf_counter()
gradient_boosting.fit(X_ada_train, y_ada_train)
minutes, seconds = divmod(round(perf_counter() - start), 60)
print(f"Fit time: {minutes} min {seconds:02d} sec")

actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)
results = []

for name, model in [
    ("Random Forest", forest),
    ("Tuned AdaBoost", tuned_adaboost),
    ("Gradient Boosting", gradient_boosting),
]:
    predicted = model.predict(X_val)
    absolute_error = np.abs(actual - predicted)
    predicted_bands = np.digitize(predicted, cutoffs)

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(predicted_bands == actual_bands),
    })

comparison = pd.DataFrame(results).set_index("Model").sort_values("MAE (£)")
display(comparison.round(3))

Training Gradient Boosting on 60,000 listings...
      Iter       Train Loss   Remaining Time 
         1           0.6334           36.07s
         2           0.5372           33.55s
         3           0.4579           33.08s
         4           0.3932           32.24s
         5           0.3388           31.15s
         6           0.2946           29.95s
         7           0.2572           28.95s
         8           0.2263           28.16s
         9           0.2008           28.11s
        10           0.1791           27.49s
        20           0.0836           20.98s
        30           0.0604           15.12s
        40           0.0519            9.57s
        50           0.0472            4.60s
        60           0.0445            0.00s
Fit time: 0 min 27 sec


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.5s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Random Forest,1850.264,24304.879,0.531,68.072,93.451
Tuned AdaBoost,2630.020,19413.520,0.701,52.062,90.242
Gradient Boosting,3093.858,26370.788,0.448,46.633,89.588


In [10]:
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

# Keep identical model inputs together during tuning.
groups = pd.util.hash_pandas_object(X_ada_train, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
fit_idx, tune_idx = next(
    splitter.split(X_ada_train, y_ada_train, groups=groups)
)

X_fit, y_fit = X_ada_train.iloc[fit_idx], y_ada_train.iloc[fit_idx]
X_tune, y_tune = X_ada_train.iloc[tune_idx], y_ada_train.iloc[tune_idx]

# Each trial inherits any settings it doesn't explicitly change.
trials = {
    "Original settings": {},
    "Deeper trees": {"max_depth": 7},
    "Smaller leaves": {"min_samples_leaf": 5},
    "More trees": {"n_estimators": 100},
    "Lower learning rate + more trees": {
        "learning_rate": 0.05,
        "n_estimators": 120,
    },
}

tuning_rows = []
params_by_trial = {}

for name, settings in trials.items():
    params = {
        f"regressor__model__{key}": value
        for key, value in settings.items()
    }
    params_by_trial[name] = params

    candidate = clone(gradient_boosting).set_params(
        regressor__model__verbose=0,
        **params,
    )

    print(f"Starting {name}...", flush=True)
    start = perf_counter()
    candidate.fit(X_fit, y_fit)
    mae = mean_absolute_error(y_tune, candidate.predict(X_tune))
    elapsed = perf_counter() - start

    tuning_rows.append({
        "Trial": name,
        "Tuning MAE (£)": mae,
        "Fit + score (sec)": elapsed,
    })
    print(f"{name}: £{mae:,.2f} MAE in {elapsed:.1f}s", flush=True)

tuning_results = pd.DataFrame(tuning_rows).sort_values("Tuning MAE (£)")
display(tuning_results.round(3))

best_name = tuning_results.iloc[0]["Trial"]
best_params = params_by_trial[best_name]
print(f"Selected trial: {best_name}")

tuned_gradient_boosting = gradient_boosting

if best_params:
    tuned_gradient_boosting = clone(gradient_boosting).set_params(
        **best_params
    )
    print("Refitting on the full Gradient Boosting training sample...", flush=True)
    tuned_gradient_boosting.fit(X_ada_train, y_ada_train)

# Compare all models on the same validation listings.
actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, cutoffs)
validation_rows = []

for name, model in [
    ("Random Forest", forest),
    ("Tuned AdaBoost", tuned_adaboost),
    ("Original Gradient Boosting", gradient_boosting),
    ("Tuned Gradient Boosting", tuned_gradient_boosting),
]:
    predicted = model.predict(X_val)
    error = np.abs(actual - predicted)

    validation_rows.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(error <= 0.10 * actual),
        "Band accuracy (%)": 100 * np.mean(
            np.digitize(predicted, cutoffs) == actual_bands
        ),
    })

comparison = pd.DataFrame(validation_rows).set_index("Model")
display(comparison.sort_values("MAE (£)").round(3))

mae_gain = (
    comparison.loc["Original Gradient Boosting", "MAE (£)"]
    - comparison.loc["Tuned Gradient Boosting", "MAE (£)"]
)
print(f"Gradient Boosting validation MAE improvement: {mae_gain:+,.2f} GBP")

Starting Original settings...
Original settings: £3,032.50 MAE in 20.2s
Starting Deeper trees...
Deeper trees: £2,607.33 MAE in 34.9s
Starting Smaller leaves...
Smaller leaves: £3,015.69 MAE in 19.6s
Starting More trees...
More trees: £2,793.11 MAE in 30.1s
Starting Lower learning rate + more trees...
Lower learning rate + more trees: £3,027.43 MAE in 39.7s


,Trial,Tuning MAE (£),Fit + score (sec)
1,Deeper trees,2607.333,34.928
3,More trees,2793.107,30.141
2,Smaller leaves,3015.691,19.566
4,Lower learning rate + more trees,3027.431,39.741
0,Original settings,3032.499,20.207


Selected trial: Deeper trees
Refitting on the full Gradient Boosting training sample...
      Iter       Train Loss   Remaining Time 
         1           0.6256            1.06m
         2           0.5239            1.09m
         3           0.4405            1.03m
         4           0.3730           59.44s
         5           0.3166           57.32s
         6           0.2705           56.20s
         7           0.2326           54.32s
         8           0.2012           52.56s
         9           0.1755           50.80s
        10           0.1542           49.45s
        20           0.0631           37.49s
        30           0.0437           26.56s
        40           0.0371           16.93s
        50           0.0341            7.87s
        60           0.0323            0.00s


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  40 out of  40 | elapsed:    0.6s finished


,MAE (£),RMSE (£),R²,Within 10% (%),Band accuracy (%)
Model,,,,,
Random Forest,1850.264,24304.879,0.531,68.072,93.451
Tuned AdaBoost,2630.020,19413.520,0.701,52.062,90.242
Tuned Gradient Boosting,2699.983,25525.213,0.483,52.405,91.002
Original Gradient Boosting,3093.858,26370.788,0.448,46.633,89.588


Gradient Boosting validation MAE improvement: +393.87 GBP
